In [1]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    # Mount Google Drive to persist the datasets and cloned repository
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing base dependencies...")
    os.system('pip install -q pandas scikit-learn transformers torch evaluate datasets accelerate peft')
    os.system('pip uninstall -y torchvision')
    print("Setup complete!")


In [2]:
# NOTE: Ensure you have `transformers`, `torch`, `evaluate`, and `accelerate` installed.
finetune_dir = 'datasets/finetuning'
output_model_dir = 'models/finetuned/xlm-roberta-base-langid'
model_name = "papluca/xlm-roberta-base-language-detection"
batch_size = 4
learning_rate = 2e-5
num_epochs = 10


In [3]:
import os
import json
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification

# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

# Our target 10 languages
TARGET_LANGUAGES = {
    "eng": "en", "sin": "si", "san": "sa", "tam": "ta", "hin": "hi", "ben": "bn", "arb": "ar", "fra": "fr", "deu": "de", "pli": "pi",
    "jpn": "ja", "nld": "nl", "pol": "pl", "ita": "it", "por": "pt", "tur": "tr", "spa": "es", "ell": "el", "urd": "ur", "bul": "bg", "cmn": "zh", "rus": "ru", "tha": "th", "swh": "sw", "vie": "vi"
}













print("Loading original model configuration...")
config = AutoConfig.from_pretrained(model_name)

# Add our new labels to the config if they don't exist
added_labels = []
for old_code, new_code in TARGET_LANGUAGES.items():
    if new_code not in config.label2id:
        idx = len(config.label2id)
        config.label2id[new_code] = idx
        config.id2label[idx] = new_code
        added_labels.append(new_code)

print(f"Added {len(added_labels)} new labels: {added_labels}")
print(f"Total labels in model: {len(config.label2id)}")

def load_data(jsonl_path):
    records = []
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            rec = json.loads(line)
            # Map our 3-letter codes to the 2-letter codes expected by the model
            mapped = TARGET_LANGUAGES.get(rec['label'], rec['label'])
            if mapped in config.label2id:
                records.append({
                    "text": rec["text"],
                    "label": config.label2id[mapped]
                })
    return pd.DataFrame(records)

print("\nLoading datasets...")
train_df = load_data(os.path.join(finetune_dir, "train_mixed.jsonl"))
val_mixed_df = load_data(os.path.join(finetune_dir, "val_mixed.jsonl"))

print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(val_mixed_df)}")


/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/ConLID/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading original model configuration...
Added 5 new labels: ['si', 'sa', 'ta', 'bn', 'pi']
Total labels in model: 25

Loading datasets...
Train size: 108218
Validation size: 17986


In [4]:
from datasets import Dataset as HFDataset

print("Tokenizing datasets...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_dataset = HFDataset.from_pandas(train_df)
val_dataset = HFDataset.from_pandas(val_mixed_df)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

# Format for PyTorch
tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_val = tokenized_val.remove_columns(["text"])
tokenized_train.set_format("torch")
tokenized_val.set_format("torch")


Tokenizing datasets...


Map: 100%|██████████| 17986/17986 [00:01<00:00, 16609.54 examples/s]


In [5]:
print("Loading model and expanding classification head...")
model = AutoModelForSequenceClassification.from_pretrained(model_name)

old_out_features = model.classifier.out_proj.out_features
new_out_features = len(config.label2id)

if new_out_features > old_out_features:
    print(f"Expanding classification head from {old_out_features} to {new_out_features} classes...")
    new_out_proj = torch.nn.Linear(model.classifier.out_proj.in_features, new_out_features)
    
    # Copy old weights
    new_out_proj.weight.data[:old_out_features] = model.classifier.out_proj.weight.data
    new_out_proj.bias.data[:old_out_features] = model.classifier.out_proj.bias.data
    
    # Initialize new weights safely
    torch.nn.init.xavier_uniform_(new_out_proj.weight.data[old_out_features:])
    torch.nn.init.zeros_(new_out_proj.bias.data[old_out_features:])
    
    model.classifier.out_proj = new_out_proj
    model.num_labels = new_out_features
    model.config = config


from peft import get_peft_model, LoraConfig, TaskType
import torch.distributed.tensor

print("Applying LoRA to freeze base weights and inject adapters...")
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=256,
    lora_alpha=512,
    lora_dropout=0.1,
    # target query and value attention matrices
    target_modules=["query", "key", "value", "dense"], 
    # train the expanded classification head
    modules_to_save=["classifier"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Model ready for finetuning.")


Loading model and expanding classification head...
Expanding classification head from 20 to 25 classes...
Applying LoRA to freeze base weights and inject adapters...
trainable params: 43,077,145 || all params: 321,140,018 || trainable%: 13.4138
Model ready for finetuning.


In [6]:
import evaluate
import numpy as np
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

# We use Micro F1 as requested by the user
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels, average="micro")

training_args = TrainingArguments(
    output_dir=output_model_dir,
    eval_strategy="epoch",  # Evaluate every epoch
    save_strategy="epoch",
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=16 // batch_size,
    fp16=torch.cuda.is_available(),
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    load_best_model_at_end=True, # Critical for Early Stopping
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none" # Disable wandb/tensorboard for simplicity
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] # Stop if F1 drops for 2 consecutive epochs
)

print("Starting Fine-tuning...")
trainer.train()

print(f"Saving final model to {output_model_dir}...")
trainer.save_model(output_model_dir)
tokenizer.save_pretrained(output_model_dir)
print("Finetuning Complete!")


Using the latest cached version of the module from /home/vihanga/.cache/huggingface/modules/evaluate_modules/metrics/evaluate-metric--f1/34c46321f42186df33a6260966e34a368f14868d9cc2ba47d142112e2800d233 (last modified on Mon Aug 10 09:12:27 2026) since it couldn't be found locally at evaluate-metric--f1, or remotely on the Hugging Face Hub.


Starting Fine-tuning...


  1%|          | 500/67630 [02:27<5:30:13,  3.39it/s]

{'loss': 0.2136, 'grad_norm': 0.5711236000061035, 'learning_rate': 1.9853319532751737e-05, 'epoch': 0.07}


  1%|▏         | 1000/67630 [04:55<5:26:49,  3.40it/s]

...

{'loss': 0.0051, 'grad_norm': 0.002450987696647644, 'learning_rate': 1.0097885553748337e-05, 'epoch': 4.95}


                                                         
 50%|█████     | 33817/67630 [2:56:22<2:46:51,  3.38it/s]/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/ConLID/.venv/lib/python3.10/site-packages/peft/utils/other.py:1496: UserWarning: Unable to fetch remote file due to the following error (MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /papluca/xlm-roberta-base-language-detection/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: de38a700-90f5-4a64-a729-a4335c2c9d21)') - silently ignoring the lookup for the file config.json in papluca/xlm-roberta-base-language-detection.
  warnings.warn(
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for

{'eval_loss': 0.07803426682949066, 'eval_f1': 0.9754809296119203, 'eval_runtime': 114.991, 'eval_samples_per_second': 156.412, 'eval_steps_per_second': 39.107, 'epoch': 5.0}


 50%|█████     | 33817/67630 [2:56:22<2:56:21,  3.20it/s]
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark/ConLID/.venv/lib/python3.10/site-packages/peft/utils/other.py:1496: UserWarning: Unable to fetch remote file due to the following error (MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /papluca/xlm-roberta-base-language-detection/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno -3] Temporary failure in name resolution)"))'), '(Request ID: bb8377d7-4792-4a72-95c0-6c20864913af)') - silently ignoring the lookup for the file config.json in papluca/xlm-roberta-base-language-detection.
  warnings.warn(
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/models/benchmark

{'train_runtime': 10582.5545, 'train_samples_per_second': 102.261, 'train_steps_per_second': 6.391, 'train_loss': 0.020070297462053223, 'epoch': 5.0}
Saving final model to models/finetuned/xlm-roberta-base-langid...
Finetuning Complete!
